# EXP002 step-1875 frozen test — merge and validate


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
REPO_REV = '158061b'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.workflow import find_prepared_data, restore_evaluation_shards
prepared = find_prepared_data('/kaggle/input')
os.environ['SPIDER_DATA_DIR'] = str(prepared)
labels = ['sft-final-shard-00-of-08', 'sft-final-shard-01-of-08', 'sft-final-shard-02-of-08', 'sft-final-shard-03-of-08', 'sft-final-shard-04-of-08', 'sft-final-shard-05-of-08', 'sft-final-shard-06-of-08', 'sft-final-shard-07-of-08']
restored = restore_evaluation_shards(['/kaggle/input'], labels, REPO_ROOT)
print({'event': 'final_test_shards_restored', 'shards': [str(path) for path in restored]})


In [ ]:
from spider.merge import merge_evaluation_shards
predictions_path, metrics = merge_evaluation_shards(
    'configs/experiment2.yaml', 'sft-final-step-1875', labels,
    ['molmoweb', 'screenspot'], 'test'
)
completed = sum(1 for _ in predictions_path.open(encoding='utf-8'))
assert completed == 5272, completed
print({'event': 'final_test_merge_complete', 'completed': completed})
metrics
